# USV neuronal coactivity analyses

This notebook quantifies how coordinated a PAG (or other region) neural population is during one class of ultrasonic vocalizations (USVs) versus another, using the pairwise spike-count correlation, population-vector cosine similarity, and population-vector Pearson correlation computed by `usv_playpen.analyses.neuronal_coactivity_engine`.

The workflow is: pooled trial-count bootstrap to a matched N, a chained circular-shuffle null per group, and a direct group-A-vs-group-B label-permutation test — reported as summary tables, per-metric null-distribution plots, a per-session breakdown, and a cross-animal slope plot.

The cells are organised as *(1) imports*, *(2) parameters — every tweakable knob*, *(3) setup & data load*, then *(4) per-section compute + plot*. Each compute/plot cell consumes the **Imports**, **Parameters**, and **Setup & load** cells; re-run those first.

The statistics and figures are factored out of the notebook: the compute lives in `usv_playpen.analyses.neuronal_coactivity_engine` (each section calls `run_group_comparison`, `compare_groups`, `pool_group_count_matrices` or `per_session_group_metrics`) and the plots + printed tables in `usv_playpen.visualizations.make_coactivity_figures`, so every section below is a short call into those two modules.

In [ ]:
### Imports

import json
import pathlib

import numpy as np
import scipy.stats as st

from usv_playpen.os_utils import configure_path
from usv_playpen.visualizations.plot_style import apply_plot_style
import usv_playpen.analyses.neuronal_coactivity_engine as engine
from usv_playpen.visualizations.make_coactivity_figures import (
    plot_acoustic_confound,
    plot_amplitude_stratified,
    plot_cross_animal_slope,
    plot_null_distributions,
    plot_per_session_pop_corr,
    summarize_acoustic_confound,
    summarize_amplitude_stratified,
    summarize_group_comparison,
)

apply_plot_style()

# Load the analyses settings once; the Parameters cell pulls the coactivity
# hyperparameters (seeds, iteration counts, window, shift range, acoustic-STFT
# band) from its `neuronal_coactivity` block rather than hard-coding them.
with open(pathlib.Path.cwd().parent / "_parameter_settings" / "analyses_settings.json", 'r') as analyses_settings_file:
    analyses_settings = json.load(analyses_settings_file)

## Parameters

Every knob a user might tweak for a run lives in the single cell below — trial segmentation, unit filters, data paths, the animal→sessions map, the coactivity hyperparameters, and the per-group plotting colours. Edit here; nothing downstream redefines these. Paths are written `/mnt/falkner/...` and wrapped in `configure_path()` so they resolve on macOS (`/Volumes/falkner`) too.

In [ ]:
### Parameters — every user-tweakable knob lives here

# Segmentation configuration
CATEGORY_COLUMN = "qlvm_supercategory"
GROUP_A_IDS   = [1]
GROUP_A_LABEL = "complex"
GROUP_B_IDS   = [7]
GROUP_B_LABEL = "simple"

# Unit-filter configuration (3 criteria: cluster_group + somatic + brain area)
CATALOG_PATH       = pathlib.Path(configure_path("/mnt/falkner/Bartul/EPHYS/unit_catalog.csv"))
UNIT_BRAIN_AREAS   = {"PAG"}
UNIT_REQUIRE_SOMATIC = True
UNIT_CLUSTER_GROUP = "good"

# Animal -> sessions mapping
# Sessions can span multiple recording days. Kilosort is run per day,
# so cluster IDs aren't stable across days; the loader picks the
# single-day block with the MOST units passing the catalog filters,
# breaking ties by session count. Days where the probe wasn't in the
# requested brain area (post-hoc histology) are automatically skipped.
DATA_ROOT = pathlib.Path(configure_path("/mnt/falkner/Bartul/Data"))

ANIMALS_TO_SESSIONS: dict[str, list[str]] = {
    "158112_0": [
        "20241107_135544", "20241107_143336", "20241107_154636",
        "20241107_172830",
    ],
    "158114_2": [
        "20241115_165038", "20241115_172055", "20241115_185527",
        "20241115_192700",
    ],
    "164335_0": [
        "20250909_151929", "20250909_154745",
        "20250911_154739", "20250911_162235", "20250911_165912",
        "20250911_182910", "20250911_190415", "20250911_194127",
        "20250912_125331", "20250912_150922", "20250912_155546",
        "20250912_174643", "20250912_181920", "20250912_185338",
        "20250913_143737", "20250913_150744", "20250913_153700",
        "20250913_173732", "20250913_180808", "20250913_183548",
    ],
    "181316_0": [
        "20250919_152924", "20250919_155842", "20250919_163351",
        "20250923_162249", "20250923_171803", "20250923_174642",
        "20250923_184326", "20250923_200538", "20250923_203320",
    ],
    "178621_2": [
        "20250927_142335", "20250927_145144", "20250927_151825",
        "20250927_172337", "20250927_175128", "20250927_181936",
        "20250928_172408", "20250928_175135", "20250928_182348",
        "20250928_192858", "20250928_195647", "20250928_202420",
    ],
    "181321_1": [
        "20251003_140135", "20251003_143416", "20251003_150452",
        "20251003_161312", "20251003_164534", "20251003_171558",
        "20251004_154544", "20251004_162927", "20251004_170028",
        "20251004_180923", "20251004_183841", "20251004_190735",
    ],
    "181322_2": [
        "20251011_140347", "20251011_145914", "20251011_153450",
        "20251011_174924", "20251011_182556", "20251011_190000",
        "20251012_150953", "20251012_154216", "20251012_161241",
        "20251012_171752", "20251012_174634", "20251012_181640",
    ],
}

CHOSEN_ANIMAL = "178621_2"

# Coactivity hyperparameters — loaded from the `neuronal_coactivity` block of
# analyses_settings.json (seeds, iteration counts, snippet window, circular-shift
# range, amplitude-stratification bins, and the acoustic-STFT band) so they are
# configured in one place rather than hard-coded here. Edit the JSON to retune.
COACTIVITY_SETTINGS = analyses_settings['neuronal_coactivity']
SEED = COACTIVITY_SETTINGS['seed']   # base RNG seed; each stochastic routine draws an independent, reproducible stream as SEED + offset
USV_BOOTSTRAP_NUM = COACTIVITY_SETTINGS['bootstrap_n']
N_BOOT_ITERATIONS = COACTIVITY_SETTINGS['n_boot']
N_SHUFFLES = COACTIVITY_SETTINGS['n_shuffles']
N_PERMUTATIONS = COACTIVITY_SETTINGS['n_permutations']
WINDOW_S = COACTIVITY_SETTINGS['window_s']
PER_SESSION_N_SHUFFLES = COACTIVITY_SETTINGS['per_session_n_shuffles']   # smaller than the chained null since we run per session
MIN_SHIFT_S = COACTIVITY_SETTINGS['min_shift_s']
MAX_SHIFT_S = COACTIVITY_SETTINGS['max_shift_s']
N_AMPLITUDE_BINS = COACTIVITY_SETTINGS['n_amplitude_bins']
MIN_BIN_TRIALS = COACTIVITY_SETTINGS['min_bin_trials']   # required per group, per bin, for a bin to be compared
ACOUSTICS_PARAMS = COACTIVITY_SETTINGS['acoustics']   # forwarded to engine.extract_snippet_acoustics (STFT band / window)

# Group plotting colours (hex). Group A uses crimson, group B uses dodgerblue
# by default — change here to retune. Labels come from the segmentation config above.
GROUP_A_COLOR = "#DC143C"
GROUP_B_COLOR = "#1E90FF"
NULL_COLOR    = "#808080"
THRESHOLD_COLOR = "#000000"

## Setup & load data

Loads the chosen animal's data through `engine.load_unit_catalog` + `engine.load_animal_sessions` — the unit-catalog read, the three-criteria unit filter (`cluster_group` + `somatic` + `brain_area`), the single-best-day population selection (Kilosort is per-day, so units aren't comparable across days), and the per-session category split now all live in `neuronal_coactivity_engine`. You should not need to edit this cell; change inputs in **Parameters** above.

In [ ]:
### Setup & load data

catalog = engine.load_unit_catalog(CATALOG_PATH)

print(f"Trial split:  `{CATEGORY_COLUMN}`")
print(f"  group A ({GROUP_A_LABEL}) = IDs {GROUP_A_IDS}")
print(f"  group B ({GROUP_B_LABEL}) = IDs {GROUP_B_IDS}")
print(f"Unit filter:  cluster_group='{UNIT_CLUSTER_GROUP}'  "
      f"somatic={UNIT_REQUIRE_SOMATIC}  brain_area in {sorted(UNIT_BRAIN_AREAS) or 'ANY'}")
print(f"Chosen animal: {CHOSEN_ANIMAL}")

sessions_data = engine.load_animal_sessions(
    CHOSEN_ANIMAL, ANIMALS_TO_SESSIONS[CHOSEN_ANIMAL],
    data_root=DATA_ROOT, catalog=catalog,
    category_column=CATEGORY_COLUMN, group_a_ids=GROUP_A_IDS, group_b_ids=GROUP_B_IDS,
    cluster_group=UNIT_CLUSTER_GROUP, require_somatic=UNIT_REQUIRE_SOMATIC,
    brain_areas=UNIT_BRAIN_AREAS,
)
n_common = len(next(iter(sessions_data))['neural_data']) if sessions_data else 0
print(f"Loaded {len(sessions_data)} sessions for {CHOSEN_ANIMAL}; common filtered units = {n_common}")

## Acoustic confound check

The fixed 30 ms window equalises call **duration**, but the two categories could still differ in the **acoustics** of that window — chiefly loudness and pitch — which might drive population activity independently of call identity. For every call this section reads the loudest-channel (`peak_amp_ch`) waveform snippet `[onset, onset + WINDOW_S)` from the session's `int16` audio and computes four features via `engine.extract_snippet_acoustics`: absolute **RMS amplitude** (from the raw waveform) plus **mean / peak frequency** and **frequency bandwidth** (energy-weighted, from the snippet's linear-power spectrogram — see the note below). It then compares the pooled `complex`-vs-`simple` distributions per feature (Mann–Whitney U + Cohen's *d*).

This is **diagnostic only** — it establishes *which* features differ between the groups (and by how much) so a control can be chosen afterwards. The headline coactivity metric `pop_corr` is mean-centred and already removes per-trial firing rate, so a future control mainly needs to guard against acoustic effects on coactivity *structure*.

**Note on the frequency features:** `mean_freq` / `freq_bandwidth` are computed as energy-weighted (linear-power) spectral centroid / spread — the textbook definitions — rather than on the cohort's `power_to_db(ref=max)` + min-max-normalised spectrogram. The cohort form relies on a per-USV region mask to suppress the noise floor; these maskless 30 ms snippets have none, so linear power gives a sharper, physically meaningful measure (`peak_freq` is identical either way). RMS is an absolute, raw-waveform measure.

In [ ]:
### Acoustic confound check: are the groups matched in amplitude & frequency?

# Features checked + their axis labels (kept here as this section's only knobs).
ACOUSTIC_FEATURES = ["rms", "mean_freq_hz", "peak_freq_hz", "freq_bandwidth_hz"]
ACOUSTIC_LABELS = {
    "rms":               "RMS amplitude (a.u.)",
    "mean_freq_hz":      "mean frequency (Hz)",
    "peak_freq_hz":      "peak frequency (Hz)",
    "freq_bandwidth_hz": "frequency bandwidth (Hz)",
}

# Pool the per-call features across the chosen animal's sessions, per group (the per-call
# loudest-channel 30 ms extraction lives in engine.compute_group_acoustics). The STFT band /
# window come from ACOUSTICS_PARAMS (the neuronal_coactivity.acoustics settings block).
group_a_acoustics = {feature: [] for feature in ACOUSTIC_FEATURES}
group_b_acoustics = {feature: [] for feature in ACOUSTIC_FEATURES}
for sess in sessions_data:
    a_feats = engine.compute_group_acoustics(sess, "group_a_df", WINDOW_S, acoustics_params=ACOUSTICS_PARAMS)
    b_feats = engine.compute_group_acoustics(sess, "group_b_df", WINDOW_S, acoustics_params=ACOUSTICS_PARAMS)
    for feature in ACOUSTIC_FEATURES:
        group_a_acoustics[feature].append(a_feats[feature])
        group_b_acoustics[feature].append(b_feats[feature])
group_a_acoustics = {f: np.concatenate(v) if v else np.array([]) for f, v in group_a_acoustics.items()}
group_b_acoustics = {f: np.concatenate(v) if v else np.array([]) for f, v in group_b_acoustics.items()}

# Per-feature Mann-Whitney U + Cohen's d table, then overlaid per-feature density histograms.
print(summarize_acoustic_confound(
    group_a_acoustics, group_b_acoustics,
    features=ACOUSTIC_FEATURES, chosen_animal=CHOSEN_ANIMAL,
    label_a=GROUP_A_LABEL, label_b=GROUP_B_LABEL,
))
plot_acoustic_confound(
    group_a_acoustics, group_b_acoustics,
    features=ACOUSTIC_FEATURES, feature_labels=ACOUSTIC_LABELS, chosen_animal=CHOSEN_ANIMAL,
    label_a=GROUP_A_LABEL, label_b=GROUP_B_LABEL,
    group_a_color=GROUP_A_COLOR, group_b_color=GROUP_B_COLOR,
)

## Compute coactivity metrics: group A vs group B

`engine.run_group_comparison` runs the full single-animal pipeline in one call — pool per-session count matrices → matched-N pooled bootstrap of each group + a direct label-permutation test → a chained circular-shuffle null per group → the per-session observed-metric breakdown — with every stochastic step seeded from `SEED`. `summarize_group_comparison` then prints the per-session deltas, each group vs its chained null, and the direct group-A-vs-group-B permutation test.

In [ ]:
### Compute coactivity metrics: group A vs group B

# Full single-animal pipeline in one call: pool per-session count matrices -> matched-N
# pooled bootstrap of each group + a direct label-permutation test -> chained
# circular-shuffle null per group -> per-session observed-metric breakdown. Every
# stochastic step derives an independent, reproducible stream from `SEED`.
results = engine.run_group_comparison(
    sessions_data,
    window_s=WINDOW_S,
    bootstrap_n=USV_BOOTSTRAP_NUM,
    n_boot=N_BOOT_ITERATIONS,
    n_shuffles=N_SHUFFLES,
    n_permutations=N_PERMUTATIONS,
    min_shift_s=MIN_SHIFT_S,
    max_shift_s=MAX_SHIFT_S,
    seed=SEED,
)

# Per-session deltas, each group vs its chained null, and the direct A-vs-B permutation test.
print(summarize_group_comparison(results, label_a=GROUP_A_LABEL, label_b=GROUP_B_LABEL))

## Plot per-metric chained-null distributions with observed bootstrap means

`plot_null_distributions` draws a 3-metric × 2-group grid from the `results` object: each group's chained-null histogram overlaid with its observed pooled-bootstrap mean and the null's 99th percentile.

In [ ]:
### Plot per-metric chained-null distributions with observed bootstrap means

plot_null_distributions(
    results,
    category_column=CATEGORY_COLUMN, group_a_ids=GROUP_A_IDS, group_b_ids=GROUP_B_IDS,
    label_a=GROUP_A_LABEL, label_b=GROUP_B_LABEL,
    group_a_color=GROUP_A_COLOR, group_b_color=GROUP_B_COLOR,
    null_color=NULL_COLOR, threshold_color=THRESHOLD_COLOR,
)

## Per-session pop_corr — how the metric behaves session-by-session

`engine.per_session_group_metrics(..., n_shuffles=PER_SESSION_N_SHUFFLES)` returns, per session, each group's observed `pop_corr` plus a within-session circular-shuffle null (both groups' onsets pooled, so the null reflects neural-timing shifts); `plot_per_session_pop_corr` draws one panel per session. Sessions with < 2 trials in either group are dropped.

In [ ]:
### Per-session pop_corr — how the metric behaves session-by-session

# Per-session observed pop_corr for both groups against a within-session circular-shuffle
# null (both groups' onsets pooled, so the null reflects neural-timing shifts). Sessions
# with < 2 trials in either group are dropped.
per_session_rows = engine.per_session_group_metrics(
    sessions_data, WINDOW_S, n_shuffles=PER_SESSION_N_SHUFFLES,
    min_shift_s=MIN_SHIFT_S, max_shift_s=MAX_SHIFT_S, seed=SEED + 100,
)
plot_per_session_pop_corr(
    per_session_rows,
    chosen_animal=CHOSEN_ANIMAL, category_column=CATEGORY_COLUMN,
    label_a=GROUP_A_LABEL, label_b=GROUP_B_LABEL,
    group_a_color=GROUP_A_COLOR, group_b_color=GROUP_B_COLOR,
    null_color=NULL_COLOR, threshold_color=THRESHOLD_COLOR,
)

## Cross-animal pop_corr summary

Loops over every focal animal in `ANIMALS_TO_SESSIONS`, loading its single best day and pooling + comparing via `engine.pool_group_count_matrices` + `engine.compare_groups`; `plot_cross_animal_slope` then draws the per-animal slope from `pop_corr(group A)` to `pop_corr(group B)`, coloured by the two-tailed permutation significance.

In [ ]:
### Cross-animal pop_corr summary

# Per animal: load its best-day sessions, pool the count matrices, and run the matched-N
# bootstrap + label-permutation comparison (engine.compare_groups). Collect the per-animal
# pop_corr means + two-tailed permutation p for the slope plot.
cross_animal_results: dict[str, dict] = {}
for animal_idx, (animal_id, session_names) in enumerate(ANIMALS_TO_SESSIONS.items()):
    print(f"Animal {animal_id} ({len(session_names)} sessions) ...", flush=True)
    animal_sessions = engine.load_animal_sessions(
        animal_id, session_names,
        data_root=DATA_ROOT, catalog=catalog,
        category_column=CATEGORY_COLUMN, group_a_ids=GROUP_A_IDS, group_b_ids=GROUP_B_IDS,
        cluster_group=UNIT_CLUSTER_GROUP, require_somatic=UNIT_REQUIRE_SOMATIC,
        brain_areas=UNIT_BRAIN_AREAS,
    )
    if not animal_sessions:
        print("  no sessions loaded, skipping")
        continue
    pooled_a, pooled_b = engine.pool_group_count_matrices(animal_sessions, WINDOW_S)
    if pooled_a.shape[1] < 1 or pooled_b.shape[1] < 1:
        print("  insufficient trials, skipping")
        continue
    bootstrap_target = min(USV_BOOTSTRAP_NUM, pooled_a.shape[1], pooled_b.shape[1])
    comparison = engine.compare_groups(
        pooled_a, pooled_b, bootstrap_n=bootstrap_target,
        n_boot=N_BOOT_ITERATIONS, n_permutations=N_PERMUTATIONS, seed=SEED + 200 + 3 * animal_idx,
    )
    pop_a_obs = float(np.mean(comparison['boot_a']['pop_corr']))
    pop_b_obs = float(np.mean(comparison['boot_b']['pop_corr']))
    perm = comparison['perm']['pop_corr']
    cross_animal_results[animal_id] = {
        'n_sessions': len(animal_sessions),
        'n_a': pooled_a.shape[1], 'n_b': pooled_b.shape[1], 'n_units': pooled_a.shape[0],
        'pop_a': pop_a_obs, 'pop_b': pop_b_obs,
        'p_two': perm['p_two_tailed'], 'p_a_gt_b': perm['p_a_gt_b'], 'z': perm['z_score'],
    }
    print(
        f"  units={pooled_a.shape[0]:>3}  n_a={pooled_a.shape[1]:>5}  n_b={pooled_b.shape[1]:>5}"
        f"  pop_a={pop_a_obs:+.4f}  pop_b={pop_b_obs:+.4f}"
        f"  Δ={pop_a_obs - pop_b_obs:+.4f}  p_two={perm['p_two_tailed']:.3f}  Z={perm['z_score']:+.2f}"
    )

plot_cross_animal_slope(
    cross_animal_results,
    category_column=CATEGORY_COLUMN, group_a_ids=GROUP_A_IDS, group_b_ids=GROUP_B_IDS,
    label_a=GROUP_A_LABEL, label_b=GROUP_B_LABEL,
    group_a_color=GROUP_A_COLOR, group_b_color=GROUP_B_COLOR,
    null_color=NULL_COLOR, threshold_color=THRESHOLD_COLOR,
)

## Amplitude-stratified pop_corr

The confound check above shows the groups **differ** acoustically (complex calls are markedly louder). This section asks whether that difference actually **drives** the headline `pop_corr` effect, or is incidental. We bin all calls by their 30 ms RMS into quantile bins and, for bins holding at least `MIN_BIN_TRIALS` of **both** groups (the overlap region), bootstrap each group to a matched N and run the same `complex`-vs-`simple` label-permutation test as the main analysis — now *within* a loudness-matched bin.

Read it like this: if the `complex > simple` `pop_corr` gap **persists within bins** (where loudness is held roughly constant), loudness is **not** the explanation and the main result stands. If the gap **collapses within bins**, the effect was a loudness artifact and amplitude must be controlled directly. Because `pop_corr` is mean-centred (per-trial firing-rate magnitude removed) and PAG amplitude is plausibly a *downstream consequence* of the population command rather than an input to it, the gap surviving here is the expected — and reassuring — outcome.

In [ ]:
### Amplitude-stratified pop_corr: does the complex-vs-simple effect survive within amplitude bins?

# Stratification knobs (N_AMPLITUDE_BINS / MIN_BIN_TRIALS) come from the Parameters
# cell above (the neuronal_coactivity settings block).

# Reuse the per-call RMS from the acoustic confound check above (aligned to the same
# session + dataframe order as the pooled spike-count matrices built here).
a_rms = group_a_acoustics['rms']
b_rms = group_b_acoustics['rms']
a_counts, b_counts = engine.pool_group_count_matrices(sessions_data, WINDOW_S)
assert a_counts.shape[1] == a_rms.shape[0] and b_counts.shape[1] == b_rms.shape[0], (
    "RMS / count-matrix misalignment — re-run the acoustic confound check cell first."
)

# Quantile bin edges over the pooled finite-positive RMS of both groups.
pooled_rms = np.concatenate([a_rms, b_rms])
pooled_rms = pooled_rms[np.isfinite(pooled_rms) & (pooled_rms > 0)]
bin_edges = np.quantile(pooled_rms, np.linspace(0.0, 1.0, N_AMPLITUDE_BINS + 1))
bin_edges[-1] = np.nextafter(bin_edges[-1], np.inf)   # make the top edge inclusive

stratified_rows = []
for bin_idx in range(N_AMPLITUDE_BINS):
    lo, hi = bin_edges[bin_idx], bin_edges[bin_idx + 1]
    a_sel = np.isfinite(a_rms) & (a_rms >= lo) & (a_rms < hi)
    b_sel = np.isfinite(b_rms) & (b_rms >= lo) & (b_rms < hi)
    n_a, n_b = int(a_sel.sum()), int(b_sel.sum())
    row = {'lo': lo, 'hi': hi, 'n_a': n_a, 'n_b': n_b, 'pop_a': np.nan, 'pop_b': np.nan, 'p_two': np.nan}
    if n_a >= MIN_BIN_TRIALS and n_b >= MIN_BIN_TRIALS:
        n_match = min(n_a, n_b)   # matched N within the bin -> fair pop_corr comparison
        comparison = engine.compare_groups(
            a_counts[:, a_sel], b_counts[:, b_sel], bootstrap_n=n_match,
            n_boot=N_BOOT_ITERATIONS, n_permutations=N_PERMUTATIONS, seed=SEED + 300 + 3 * bin_idx,
        )
        row['pop_a'] = float(np.mean(comparison['boot_a']['pop_corr']))
        row['pop_b'] = float(np.mean(comparison['boot_b']['pop_corr']))
        row['p_two'] = comparison['perm']['pop_corr']['p_two_tailed']
    stratified_rows.append(row)

# Unstratified reference (all trials, matched-N bootstrap) for context.
n_overall = min(a_counts.shape[1], b_counts.shape[1], USV_BOOTSTRAP_NUM)
pop_a_overall = float(np.mean(engine.bootstrap_coactivity_distribution(a_counts, n_overall, N_BOOT_ITERATIONS, seed=SEED + 600)['pop_corr']))
pop_b_overall = float(np.mean(engine.bootstrap_coactivity_distribution(b_counts, n_overall, N_BOOT_ITERATIONS, seed=SEED + 601)['pop_corr']))

print(summarize_amplitude_stratified(
    stratified_rows, pop_a_overall, pop_b_overall,
    chosen_animal=CHOSEN_ANIMAL, n_bins=N_AMPLITUDE_BINS,
    label_a=GROUP_A_LABEL, label_b=GROUP_B_LABEL,
))
plot_amplitude_stratified(
    stratified_rows, pop_a_overall, pop_b_overall,
    chosen_animal=CHOSEN_ANIMAL, label_a=GROUP_A_LABEL, label_b=GROUP_B_LABEL,
    group_a_color=GROUP_A_COLOR, group_b_color=GROUP_B_COLOR, threshold_color=THRESHOLD_COLOR,
)